# Notebook 1: Experiment 1 — Same-Stock Prediction (80/20)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on each stock's daily data and predict its own future prices.  
**Train/Test Split:** 80/20 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Stocks:** TLKM, BBCA, ASII, UNVR  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_seed()
set_ieee_style()
check_gpu()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.8
RATIO_LABEL = '80_20'
EXP_LABEL = f'Exp1_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 1 - Same Stock Prediction (80/20)")
print(f"Train ratio: {TRAIN_RATIO}, Test ratio: {1-TRAIN_RATIO}")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 60, Epochs: 200, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

Memory growth enabled (dynamic allocation)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations

Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations

Working directory: d:\Data_D\All Programming Language\Python\0_TA\Ultimate Code
Data directory: d:\Data_D\All Programming Language\Python\0_TA\Ultimate Code\dataset
Experiment 1 - Same Stock Prediction (80/20)
Train ratio: 0.8, Test ratio: 0.19999999999999996


In [2]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


## Run All Experiments

In [3]:
# ============================================================
# EXPERIMENT 1: Train and predict on same stock
# ============================================================
all_results = []
all_predictions = {}  # {stock: {model_type: (y_true, y_pred, dates)}}
all_histories = {}    # {stock: {model_type: history}}

for stock in STOCKS:
    print(f"\n############################################################")
    print(f"# STOCK: {stock}")
    print(f"############################################################")
    
    # Prepare data
    X_train, y_train, X_test, y_test, test_dates = prepare_same_stock_data(
        daily_data[stock], train_ratio=TRAIN_RATIO, lookback=LOOKBACK
    )
    print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
    
    all_predictions[stock] = {}
    all_histories[stock] = {}
    
    for model_type in MODEL_TYPES:
        exp_name = f'{EXP_LABEL}_{stock}'
        
        y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(
            model_type=model_type,
            X_train=X_train, y_train=y_train,
            X_test=X_test, y_test=y_test,
            experiment_name=exp_name,
            save_dir=f'models/{EXP_LABEL}',
            epochs=EPOCHS, batch_size=BATCH_SIZE
        )
        
        # Store results
        result = {'Stock': stock, 'Model': model_type, **metrics}
        all_results.append(result)
        all_predictions[stock][model_type] = (y_true_inv, y_pred_inv, test_dates)
        all_histories[stock][model_type] = history
        
        # Plot individual prediction
        plot_actual_vs_predicted(
            test_dates, y_true_inv, y_pred_inv,
            model_type, stock, EXP_LABEL,
            save_dir=f'figures/{EXP_LABEL}'
        )
        
        # Plot training history
        plot_training_history(
            history, model_type, stock, EXP_LABEL,
            save_dir=f'figures/{EXP_LABEL}'
        )

print("\n\nAll Experiment 1 (80/20) training complete!")



############################################################
# STOCK: TLKM
############################################################
  X_train: (4134, 60, 1), X_test: (1049, 60, 1)

Training BiLSTM for: Exp1_80_20_TLKM
  Train samples: 4134, Test samples: 1049
Epoch 1/200
58/59 [============================>.] - ETA: 0s - loss: 0.0012
Epoch 1: val_loss improved from inf to 0.00024, saving model to models/Exp1_80_20\Exp1_80_20_TLKM_BiLSTM_best.keras
59/59 [==============================] - 13s 60ms/step - loss: 0.0012 - val_loss: 2.4443e-04
Epoch 2/200
59/59 [==============================] - ETA: 0s - loss: 1.5805e-04
Epoch 2: val_loss improved from 0.00024 to 0.00024, saving model to models/Exp1_80_20\Exp1_80_20_TLKM_BiLSTM_best.keras
59/59 [==============================] - 2s 33ms/step - loss: 1.5805e-04 - val_loss: 2.3912e-04
Epoch 3/200
59/59 [==============================] - ETA: 0s - loss: 1.4107e-04
Epoch 3: val_loss improved from 0.00024 to 0.00019, saving model to models

## Results Summary

In [4]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 1 - Same Stock Prediction (80/20)")

# Save results
results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 1 - Same Stock Prediction (80/20)
Stock  Model         MSE     RMSE      MAE  MAPE (%)       R2  Training_Time_s  Epochs_Run
 TLKM BiLSTM   6824.4987  82.6105  65.9856    2.0748 0.960052            421.1         190
 TLKM  BiGRU   6010.2834  77.5260  59.4508    1.9111 0.964819            170.0          87
 TLKM   LSTM   4598.3469  67.8111  51.1906    1.6420 0.973083            238.5         200
 TLKM    GRU   3927.6837  62.6712  46.5959    1.5129 0.977009            158.5         140
 BBCA BiLSTM  39031.6644 197.5643 160.5885    1.9585 0.963888            281.3         142
 BBCA  BiGRU  40242.0612 200.6042 156.5391    1.9323 0.962768            125.2          62
 BBCA   LSTM 102740.3058 320.5313 262.0399    3.0375 0.904944            219.6         184
 BBCA    GRU  27222.2001 164.9915 124.8147    1.5181 0.974814             33.8          27
 ASII BiLSTM  12900.7963 113.5817  85.9070    1.7988 0.965082            162.4          79
 ASII  BiGRU  10347.6517 101.7234  76.1804

## Visualizations

In [5]:
# ============================================================
# ALL MODELS COMPARISON PER STOCK
# ============================================================
for stock in STOCKS:
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    preds = {mt: all_predictions[stock][mt][1] for mt in MODEL_TYPES}
    
    plot_all_models_comparison(
        dates, y_true, preds, stock, EXP_LABEL,
        save_dir=f'figures/{EXP_LABEL}'
    )

print("All comparison plots saved!")


  Figure saved: figures/Exp1_80_20/Exp1_80_20_TLKM_all_models.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_BBCA_all_models.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_ASII_all_models.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_UNVR_all_models.png
All comparison plots saved!


In [6]:
# ============================================================
# METRICS BAR CHARTS
# ============================================================
for metric in ['MSE', 'RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_comparison_bar(
        results_df, metric, EXP_LABEL,
        group_col='Stock', save_dir=f'figures/{EXP_LABEL}'
    )

print("All metrics bar charts saved!")


  Figure saved: figures/Exp1_80_20/Exp1_80_20_MSE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_RMSE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_MAE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_MAPE_pct_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_R2_comparison.png
All metrics bar charts saved!


In [7]:
# ============================================================
# SUMMARY: BEST MODEL PER STOCK
# ============================================================
print("\n" + "="*60)
print("  BEST MODEL PER STOCK (by RMSE)")
print("="*60)
for stock in STOCKS:
    stock_results = results_df[results_df['Stock'] == stock]
    best_idx = stock_results['RMSE'].idxmin()
    best = stock_results.loc[best_idx]
    print(f"  {stock}: {best['Model']} (RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")

print("\n  BEST MODEL PER STOCK (by R² Score)")
print("="*60)
for stock in STOCKS:
    stock_results = results_df[results_df['Stock'] == stock]
    best_idx = stock_results['R2'].idxmax()
    best = stock_results.loc[best_idx]
    print(f"  {stock}: {best['Model']} (R²={best['R2']:.6f}, RMSE={best['RMSE']:.4f})")



  BEST MODEL PER STOCK (by RMSE)
  TLKM: GRU (RMSE=62.6712, R²=0.977009)
  BBCA: GRU (RMSE=164.9915, R²=0.974814)
  ASII: LSTM (RMSE=85.0905, R²=0.980403)
  UNVR: LSTM (RMSE=90.6397, R²=0.990556)

  BEST MODEL PER STOCK (by R² Score)
  TLKM: GRU (R²=0.977009, RMSE=62.6712)
  BBCA: GRU (R²=0.974814, RMSE=164.9915)
  ASII: LSTM (R²=0.980403, RMSE=85.0905)
  UNVR: LSTM (R²=0.990556, RMSE=90.6397)
